# Lab 15: Mogi Sources and Volcano Deformation

> **Colab note:** This notebook is designed to run on **Google Colab**. [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amtseismo/EPS166/blob/main/notebooks/15_volcano_deformation.ipynb)

An interactive browser-based version of Parts I and III is also available at the course website. Use it to build visual intuition before working through the quantitative exercises here.

## Introduction

The Mogi (1958) point source is the most widely used model for volcanic inflation and deflation. It predicts an axially symmetric deformation field that depends on only three parameters: source depth, volume change, and horizontal position. Despite its simplicity, it explains the first-order pattern of dozens of volcanic InSAR observations.

In this lab, you will implement the Mogi forward model in Python, generate synthetic wrapped interferograms, and invert a digitized radial LOS profile from a real ALOS-2 interferogram of Domuyo volcano, Argentina.

## Learning objectives

By the end, you will be able to:

- implement and verify the Mogi surface displacement equations
- explain how source depth controls the width of the deformation signal
- generate a synthetic wrapped interferogram and count fringes
- describe how satellite look geometry projects 3D displacement into LOS
- invert a radial LOS profile for Mogi source depth and volume change
- map the depth–volume misfit surface and characterize the trade-off

## Notebook Outline
- [Part I: The Mogi Forward Model](#part-i-the-mogi-forward-model)
- [Part II: Synthetic Wrapped Interferograms](#part-ii-synthetic-wrapped-interferograms)
- [Part III: Domuyo Inverse Problem](#part-iii-domuyo-inverse-problem)
- [Part IV: The Depth–Volume Trade-off](#part-iv-the-depthvolume-tradeoff)
- [Synthesis](#synthesis)
- [Summary](#summary)


## Setup

The notebook uses NumPy, Matplotlib, and SciPy. Run this cell once before continuing.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.optimize import minimize

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


---

## Part I: The Mogi Forward Model

The Mogi (1958) point source gives the following surface displacements at horizontal distance $r$ from a source at depth $d$ with volume change $\Delta V$ (Segall, 2010, Ch. 7):

$$u_r(r) = \frac{(1-\nu)}{\pi} \frac{\Delta V \cdot r}{(r^2 + d^2)^{3/2}}$$

$$u_z(r) = \frac{(1-\nu)}{\pi} \frac{\Delta V \cdot d}{(r^2 + d^2)^{3/2}}$$

where $\nu$ is Poisson's ratio (0.25 for typical crustal rock), $u_r$ is horizontal radial displacement (positive outward), and $u_z$ is vertical displacement (positive upward).

### Prediction

Before implementing anything: how should peak vertical displacement $u_z^{\max}$ scale with source depth $d$ if volume change $\Delta V$ is held fixed? Write an expression and explain the physical reason.


In [ ]:
def mogi_displacement(x, y, x0=0.0, y0=0.0, depth_km=5.0, dV_km3=0.05, nu=0.25):
    """
    Mogi (1958) point-source surface displacement.

    Valid when source radius a << depth d (point-source limit).
    Reference: Segall (2010), Earthquake and Volcano Deformation, Ch. 7.

    Parameters
    ----------
    x, y : array-like
        Surface coordinates in km. Can be scalar or array (e.g., meshgrid).
    x0, y0 : float
        Source horizontal position in km.
    depth_km : float
        Source depth in km (positive downward).
    dV_km3 : float
        Volume change in km³ (positive = inflation, negative = deflation).
    nu : float
        Poisson's ratio (default 0.25).

    Returns
    -------
    ue, un, uz : array
        East, north, and vertical surface displacement in meters.
    """
    dx = (x - x0) * 1e3   # convert km → m
    dy = (y - y0) * 1e3
    d  = depth_km * 1e3
    dV = dV_km3   * 1e9    # convert km³ → m³

    r2   = dx**2 + dy**2
    # (r² + d²)^(3/2)  — written as a product to avoid a slow Math.pow call
    dist = np.sqrt(r2 + d**2)
    dist3 = (r2 + d**2) * dist     # = (r² + d²)^(3/2)

    C  = (1 - nu) / np.pi * dV / dist3
    return C * dx, C * dy, C * d


### 1a. Verify the depth–width scaling

The vertical displacement at the surface peaks directly above the source:

$$u_z^{\max} = \frac{(1-\nu)\,\Delta V}{\pi\, d^2}$$

and falls to half that value at the **half-amplitude radius**:

$$r_{1/2} = d\sqrt{2^{2/3} - 1} \approx 0.77\,d$$

Compute the radial profile of $u_z$ for three source depths with the same $\Delta V$, and verify the $r_{1/2} \approx 0.77d$ relationship numerically.


In [ ]:
r_km   = np.linspace(0, 30, 300)   # horizontal distances in km
dV_km3 = 0.05                       # fixed volume change
depths = [2.0, 5.0, 10.0]           # three source depths in km

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for d in depths:
    _, _, uz = mogi_displacement(r_km, 0.0, depth_km=d, dV_km3=dV_km3)
    uz_cm = uz * 100   # convert m → cm

    # Theoretical half-amplitude radius
    r_half_theory = d * np.sqrt(2**(2/3) - 1)

    # Numerical half-amplitude radius
    uz_max = uz_cm[0]
    idx = np.argmin(np.abs(uz_cm - uz_max / 2))
    r_half_num = r_km[idx]

    axes[0].plot(r_km, uz_cm, label=f"d = {d} km (r½ ≈ {r_half_num:.1f} km)")
    axes[0].axvline(r_half_num, color=axes[0].lines[-1].get_color(), ls='--', alpha=0.5)

    print(f"d = {d:5.1f} km | theoretical r½ = {r_half_theory:.2f} km "
          f"| numerical r½ = {r_half_num:.2f} km")

axes[0].set(xlabel="Radial distance (km)", ylabel="Vertical displacement (cm)",
            title=f"Mogi radial profiles, ΔV = {dV_km3} km³")
axes[0].legend()

# Panel 2: normalized profiles
for d in depths:
    _, _, uz = mogi_displacement(r_km, 0.0, depth_km=d, dV_km3=dV_km3)
    axes[1].plot(r_km / d, uz / uz[0], label=f"d = {d} km")

axes[1].axhline(0.5, color='k', ls=':', lw=1, label="half-amplitude")
axes[1].axvline(np.sqrt(2**(2/3) - 1), color='k', ls='--', lw=1, label=f"r/d = {np.sqrt(2**(2/3)-1):.2f}")
axes[1].set(xlabel="r / d (normalized)", ylabel="u_z / u_z(0)",
            title="Normalized profiles collapse onto one curve")
axes[1].legend()

plt.tight_layout()
plt.show()


> **Interpret the profiles:**
> 1. All three normalized profiles collapse onto one curve. What does this tell you about the Mogi model?
> 2. The half-amplitude radius increases proportionally with depth. Write an equation relating $r_{1/2}$ to $d$. How could you use a measured $r_{1/2}$ to estimate source depth from an interferogram?
> 3. Holding $\Delta V$ fixed and doubling the depth, by what factor does $u_z^{\max}$ change? Show this mathematically.


### 1b. Horizontal vs. vertical displacement

The ratio $u_r / u_z = r / d$ is a key geometric result. Plot both components and find the crossover point.


In [ ]:
depth_km = 6.5
dV_km3   = 0.082

_, _, uz = mogi_displacement(r_km, 0.0, depth_km=depth_km, dV_km3=dV_km3)
# For the radial horizontal, evaluate at points along the x-axis (y=0)
# dx = r, so ue = ur (all displacement is in the x-direction)
ur, _, _ = mogi_displacement(r_km, 0.0, x0=0.0, y0=0.0,
                              depth_km=depth_km, dV_km3=dV_km3)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(r_km, uz * 100, label="Vertical $u_z$", lw=2)
ax.plot(r_km, ur * 100, label="Horizontal radial $u_r$", lw=2, ls='--')
ax.axvline(depth_km, color='gray', ls=':', label=f"r = d = {depth_km} km")
ax.set(xlabel="Radial distance (km)", ylabel="Displacement (cm)",
       title=f"Mogi source: d = {depth_km} km, ΔV = {dV_km3} km³")
ax.legend()
plt.tight_layout()
plt.show()

# Compute the crossover radius numerically
crossover_idx = np.argmin(np.abs(uz - np.abs(ur)))
print(f"Crossover radius (u_r = u_z): {r_km[crossover_idx]:.1f} km")
print(f"Source depth:                 {depth_km:.1f} km")


> **Interpret the crossover:**
> 1. At what radial distance does horizontal displacement equal vertical displacement? How does this compare to the source depth?
> 2. If you observed tilt at a station directly above the source, what would the tilt direction and magnitude tell you?
> 3. GNSS vectors at a volcano show primarily vertical uplift at stations close to the summit and increasingly outward-tilted vectors at stations farther away. What does this pattern imply about the source depth relative to station spacing?


---

## Part II: Synthetic Wrapped Interferograms

An interferogram measures LOS displacement — the projection of the 3D displacement vector onto the satellite look direction. For a satellite with incidence angle $\theta$ and look azimuth $\alpha$ (satellite-to-ground direction, clockwise from north):

$$\text{LOS} = -\sin\theta\,\sin\alpha\cdot u_E - \sin\theta\,\cos\alpha\cdot u_N + \cos\theta\cdot u_z$$

Positive LOS = ground moves toward satellite.

A **wrapped interferogram** displays the LOS phase modulo one cycle (commonly 2.8 cm for C-band Sentinel-1 or 10 cm for display). This produces the characteristic bulls-eye fringe pattern.

### Prediction

An ascending satellite looks approximately eastward (ENE). How should the interferogram for a Mogi source differ from perfect circular symmetry, and on which side — east or west of the volcano — would you expect slightly more fringes? Why?


In [ ]:
def los_displacement(ue, un, uz, incidence_deg=36.0, look_az_deg=78.0):
    """
    Project 3D surface displacement into radar LOS.

    Parameters
    ----------
    ue, un, uz : array
        East, north, and up displacements in meters.
    incidence_deg : float
        Incidence angle from vertical, in degrees.
    look_az_deg : float
        Azimuth of satellite-to-ground look direction,
        measured clockwise from north, in degrees.

    Returns
    -------
    los : array
        LOS displacement in same units as input (positive = toward satellite).
    """
    t = np.radians(incidence_deg)
    a = np.radians(look_az_deg)
    return (-np.sin(t) * np.sin(a) * ue
            -np.sin(t) * np.cos(a) * un
            + np.cos(t) * uz)


def phase_to_rgb(phase):
    """
    Convert phase in [0, 1) to interferogram color cycle.

    Color sequence: cyan (0) → yellow (1/3) → magenta (2/3) → cyan (1).
    This is the standard cycle used in volcanological InSAR display.

    Parameters
    ----------
    phase : array, shape (...)
        Phase values in [0, 1).

    Returns
    -------
    rgb : array, shape (..., 3)
        RGB values in [0, 1].
    """
    p = phase * 3.0
    r = np.where(p < 1, p,     np.where(p < 2, 1.0,   3 - p))
    g = np.where(p < 1, 1.0,   np.where(p < 2, 2 - p, p - 2))
    b = np.where(p < 1, 1 - p, np.where(p < 2, p - 1, 1.0))
    return np.clip(np.stack([r, g, b], axis=-1), 0, 1)


def synthetic_ifg(depth_km, dV_km3, x0=0.0, y0=0.0,
                  incidence_deg=36.0, look_az_deg=78.0,
                  scene_km=50, npix=400, cycle_m=0.10):
    """
    Generate a synthetic wrapped interferogram for a Mogi source.

    Parameters
    ----------
    depth_km : float
        Source depth in km.
    dV_km3 : float
        Volume change in km³.
    x0, y0 : float
        Source horizontal position in km from scene center.
    incidence_deg : float
        Radar incidence angle from vertical.
    look_az_deg : float
        Look direction azimuth (satellite-to-ground, clockwise from north).
    scene_km : float
        Scene half-width in km (scene spans –scene_km/2 to +scene_km/2).
    npix : int
        Number of pixels along each side.
    cycle_m : float
        LOS per color cycle in meters (default 0.10 = 10 cm).

    Returns
    -------
    los_rel : 2D array
        Relative LOS displacement in meters (referenced to scene corner).
    rgb : (npix, npix, 3) array
        Wrapped interferogram as RGB image.
    extent : list
        [xmin, xmax, ymin, ymax] in km, for imshow extent.
    """
    half = scene_km / 2
    xv   = np.linspace(-half, half, npix)
    yv   = np.linspace(-half, half, npix)
    xx, yy = np.meshgrid(xv, yv[::-1])  # y increases upward in image

    ue, un, uz = mogi_displacement(xx, yy, x0=x0, y0=y0,
                                   depth_km=depth_km, dV_km3=dV_km3)
    los = los_displacement(ue, un, uz, incidence_deg, look_az_deg)

    # Reference to far corner (simulates relative InSAR measurement)
    rc = half * 0.95
    ue_r, un_r, uz_r = mogi_displacement(rc, rc, x0=x0, y0=y0,
                                          depth_km=depth_km, dV_km3=dV_km3)
    los_ref = los_displacement(ue_r, un_r, uz_r, incidence_deg, look_az_deg)

    los_rel = los - los_ref
    phase   = (los_rel / cycle_m) % 1.0  # fractional phase [0, 1)
    rgb     = phase_to_rgb(phase)
    extent  = [-half, half, -half, half]
    return los_rel, rgb, extent


### 2a. Generate and display synthetic interferograms

Compare interferograms for shallow and deep sources with the same volume change.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

configs = [
    dict(depth_km=2.5,  dV_km3=0.015, label="Shallow\nd = 2.5 km, ΔV = 0.015 km³"),
    dict(depth_km=6.5,  dV_km3=0.082, label="Domuyo-like\nd = 6.5 km, ΔV = 0.082 km³"),
    dict(depth_km=12.0, dV_km3=0.300, label="Deep\nd = 12 km, ΔV = 0.300 km³"),
]

for ax, cfg in zip(axes, configs):
    los_rel, rgb, ext = synthetic_ifg(**{k: v for k, v in cfg.items() if k != 'label'})
    ax.imshow(rgb, origin='upper', extent=ext, aspect='equal')
    ax.set(title=cfg['label'], xlabel="East (km)", ylabel="North (km)")
    # Mark source location
    ax.plot(0, 0, '+w', ms=10, mew=2)
    # 10-km scale bar
    ax.plot([-20, -10], [-22, -22], 'w-', lw=2)
    ax.text(-15, -21, '10 km', color='w', ha='center', fontsize=9)

plt.suptitle("Synthetic ALOS-2 ascending interferograms (10 cm / color cycle)", y=1.01)
plt.tight_layout()
plt.show()


> **Interpret the interferograms:**
> 1. Count the approximate number of fringes from edge to center for each interferogram. How does fringe count relate to peak LOS?
> 2. Notice the slight left–right (east–west) asymmetry in fringe density. Which side has more tightly spaced fringes, and why?
> 3. For the same volume change, would a deeper source or shallower source produce more fringes? Explain using the formula for peak LOS.


### 2b. Ascending vs. descending geometry

Compare the same source as it appears in ascending (looking ENE) vs. descending (looking WNW) satellite passes.


In [ ]:
depth_km = 6.5
dV_km3   = 0.082

# ALOS-2 ascending Path 34: θ ≈ 36°, look azimuth ≈ 78° (ENE)
# ALOS-2 descending:        θ ≈ 36°, look azimuth ≈ 282° (WNW)
geoms = [
    dict(look_az_deg=78,  label="Ascending (look ENE, α = 78°)"),
    dict(look_az_deg=282, label="Descending (look WNW, α = 282°)"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, g in zip(axes, geoms):
    los_rel, rgb, ext = synthetic_ifg(depth_km=depth_km, dV_km3=dV_km3,
                                      incidence_deg=36, look_az_deg=g['look_az_deg'])
    ax.imshow(rgb, origin='upper', extent=ext, aspect='equal')
    ax.set(title=g['label'], xlabel="East (km)", ylabel="North (km)")
    ax.plot(0, 0, '+w', ms=10, mew=2)

plt.suptitle(f"Ascending vs. descending geometry, d = {depth_km} km, ΔV = {dV_km3} km³", y=1.01)
plt.tight_layout()
plt.show()

# Also plot the 1D LOS profiles along the east axis for both passes
r_km_1d = np.linspace(-25, 25, 500)

fig2, ax2 = plt.subplots(figsize=(9, 4))
for g in geoms:
    ue, un, uz = mogi_displacement(r_km_1d, 0.0, depth_km=depth_km, dV_km3=dV_km3)
    los = los_displacement(ue, un, uz, incidence_deg=36, look_az_deg=g['look_az_deg'])
    ax2.plot(r_km_1d, los * 100, label=g['label'])

ax2.axhline(0, color='k', lw=0.5)
ax2.axvline(0, color='k', lw=0.5, ls='--', label="Source position")
ax2.set(xlabel="East offset from source (km)",
        ylabel="LOS displacement (cm)",
        title="East–West LOS profiles: ascending vs. descending")
ax2.legend()
plt.tight_layout()
plt.show()


> **Interpret the ascending/descending comparison:**
> 1. The LOS profile along the east axis is not symmetric about the source for either geometry — and the asymmetry is **mirrored** between ascending and descending. Explain why using the LOS projection formula.
> 2. At large radii (r > 10 km) in the eastward direction, the ascending LOS can approach zero or become slightly negative even for a pure inflation source. What is happening physically?
> 3. If you wanted to measure the horizontal east–west displacement field, which combination of ascending and descending LOS would you form? Write the equation.


---

## Part III: Domuyo Inverse Problem

Domuyo volcano in Neuquén Province, Argentina, has been inflating at ~12 cm/yr LOS since at least mid-2014. The dataset below is a digitized radial LOS profile from the ALOS-2 ascending Path 34 interferogram spanning **7 February 2015 to 1 April 2018 (3.17 years)**.

**Satellite geometry:** ALOS-2 L-band (λ = 23.8 cm), incidence angle θ ≈ 36°, look direction ≈ ENE.

**Data:** Radial profile along the eastward direction from the volcano summit. Each point has ±2 cm measurement uncertainty (noise + residual atmospheric phase).

**Data source:** Adapted from Lundgren et al. (2020), Sci. Rep. 10, 11642 (CC BY 4.0). The published geodetic source model uses a Compound Dislocation Model (CDM — oblate ellipsoidal sill), not a Mogi sphere. Your Mogi fit is a useful approximation.

### Prediction

Before fitting: use the radial profile data to make a first-order estimate of source depth using the half-amplitude radius relationship $d \approx r_{1/2} / 0.77$. What depth do you predict?


In [ ]:
# Domuyo ALOS-2 radial LOS profile
# Span: 7 Feb 2015 – 1 Apr 2018 (3.17 years of inflation)
# Profile along eastward direction from volcano summit
# LOS approximation: cos(theta) × u_z (circularly averaged, vertical-dominant)

dom_r_km    = np.array([ 0.0,  1.5,  3.0,  4.5,  6.0,  7.5,
                          9.0, 11.0, 13.0, 15.0, 17.0, 20.0,
                         23.0, 26.0, 29.0])          # km from summit

dom_los_cm  = np.array([38.8, 32.9, 30.2, 19.9, 16.4,  8.2,
                          9.3,  3.7,  5.9,  0.8,  2.5, -0.9,
                          2.2, -0.6,  1.3])          # cm, positive = toward satellite

dom_sigma_cm = np.full(len(dom_r_km), 2.0)           # cm, 1-sigma uncertainty

# Plot the data
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(dom_r_km, dom_los_cm, yerr=dom_sigma_cm,
            fmt='o', color='#c47c18', capsize=4, label="Observed (Domuyo, 2015–2018)")
ax.axhline(0, color='k', lw=0.5)
ax.set(xlabel="Distance from summit (km)",
       ylabel="LOS displacement (cm)",
       title="Domuyo ALOS-2 radial LOS profile (3.17-year interferogram)")
ax.legend()
plt.tight_layout()
plt.show()

# First-order depth estimate from half-amplitude radius
peak_los = dom_los_cm[0]
half_amp = peak_los / 2
print(f"Peak LOS:        {peak_los:.1f} cm")
print(f"Half-amplitude:  {half_amp:.1f} cm")

# Interpolate to find the half-amplitude radius
r_half_interp = np.interp(half_amp, dom_los_cm[::-1], dom_r_km[::-1])
d_estimate    = r_half_interp / np.sqrt(2**(2/3) - 1)
print(f"r½ (interpolated): {r_half_interp:.1f} km")
print(f"Depth estimate:    {d_estimate:.1f} km")


### 3a. Build a forward model for the LOS profile

For fitting, we use the simplified LOS approximation:

$$\text{LOS}(r) \approx \cos\theta \cdot u_z(r)$$

This is valid because (1) vertical displacement dominates the LOS for typical incidence angles, and (2) a circularly-averaged radial profile averages out the azimuthal asymmetry from the horizontal component. The approximation is accurate to ~10–15% for $\theta$ = 36°.


In [ ]:
THETA_DEG = 36.0   # ALOS-2 incidence angle
SPAN_YR   = 3.17   # interferogram time span in years

def mogi_los_profile(r_km, depth_km, dV_km3, nu=0.25):
    """
    Mogi LOS profile using vertical-dominant approximation.

    LOS ≈ cos(theta) × u_z(r), valid for moderate incidence angles
    when averaging azimuthal asymmetry. Returns LOS in centimeters.

    Parameters
    ----------
    r_km : array
        Radial distances in km.
    depth_km : float
        Source depth in km.
    dV_km3 : float
        Source volume change in km³.
    nu : float
        Poisson's ratio.
    """
    _, _, uz = mogi_displacement(r_km, 0.0, depth_km=depth_km, dV_km3=dV_km3, nu=nu)
    return np.cos(np.radians(THETA_DEG)) * uz * 100   # m → cm


def residuals(params, r_km, los_cm, sigma_cm):
    """Weighted residuals for Mogi LOS profile fit."""
    depth_km, dV_km3 = params
    if depth_km <= 0 or dV_km3 <= 0:
        return np.full(len(r_km), 1e6)  # penalty for nonphysical parameters
    model = mogi_los_profile(r_km, depth_km, dV_km3)
    return (los_cm - model) / sigma_cm


def rms_misfit(depth_km, dV_km3, r_km, los_cm, sigma_cm):
    """Root-mean-square misfit in cm."""
    model = mogi_los_profile(r_km, depth_km, dV_km3)
    return np.sqrt(np.mean(((los_cm - model) / sigma_cm)**2))


### 3b. Manual forward modeling

Before running an automated inversion, try to fit the data manually by adjusting `depth_km` and `dV_km3`. Start with your first-order estimate from Part 3a. What depth and volume change give a visually good fit?


In [ ]:
# --- Adjust these parameters to fit the data ---
depth_try = 5.0    # km   ← change this
dV_try    = 0.050  # km³  ← change this
# ------------------------------------------------

r_model  = np.linspace(0, 30, 300)
los_model = mogi_los_profile(r_model, depth_try, dV_try)
rms = rms_misfit(depth_try, dV_try, dom_r_km, dom_los_cm, dom_sigma_cm)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Profile comparison
axes[0].errorbar(dom_r_km, dom_los_cm, yerr=dom_sigma_cm,
                 fmt='o', color='#c47c18', capsize=4, label="Observed", zorder=5)
axes[0].plot(r_model, los_model, '-', color='#3a4fc7', lw=2,
             label=f"Model: d={depth_try} km, ΔV={dV_try} km³")
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set(xlabel="Distance from summit (km)",
            ylabel="LOS displacement (cm)",
            title=f"RMS misfit = {rms:.2f} (normalized)")
axes[0].legend()

# Synthetic interferogram for the trial model
_, rgb, ext = synthetic_ifg(depth_km=depth_try, dV_km3=dV_try,
                             incidence_deg=36, look_az_deg=78)
axes[1].imshow(rgb, origin='upper', extent=ext, aspect='equal')
axes[1].plot(0, 0, '+w', ms=10, mew=2)
axes[1].set(title=f"Synthetic interferogram (d={depth_try} km, ΔV={dV_try} km³)",
            xlabel="East (km)", ylabel="North (km)")

plt.suptitle("Manual Mogi fit — adjust depth_try and dV_try", y=1.01)
plt.tight_layout()
plt.show()

print(f"Depth:          {depth_try:.1f} km")
print(f"Volume change:  {dV_try:.4f} km³ total ({dV_try/SPAN_YR:.4f} km³/yr)")
print(f"RMS misfit:     {rms:.3f} (normalized; < 1.0 is a good fit)")


### 3c. Nonlinear least-squares inversion

Use `scipy.optimize.minimize` to find the best-fit depth and volume change.


In [ ]:
from scipy.optimize import minimize

def objective(params):
    depth_km, dV_km3 = params
    if depth_km <= 0 or dV_km3 <= 0:
        return 1e10
    return rms_misfit(depth_km, dV_km3, dom_r_km, dom_los_cm, dom_sigma_cm)

# Starting point from your first-order estimate
x0_guess = [depth_try, dV_try]

result = minimize(objective, x0_guess, method='Nelder-Mead',
                  options={'xatol': 0.01, 'fatol': 0.001, 'maxiter': 2000})

d_best, dV_best = result.x
rms_best = result.fun

print("=== Best-fit Mogi parameters ===")
print(f"Depth:          {d_best:.2f} km")
print(f"Volume change:  {dV_best:.4f} km³ (3.17 yr total)")
print(f"Volume rate:    {dV_best/SPAN_YR:.4f} km³/yr")
print(f"RMS misfit:     {rms_best:.3f} (normalized)")

# Plot the best fit
los_best = mogi_los_profile(r_model, d_best, dV_best)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].errorbar(dom_r_km, dom_los_cm, yerr=dom_sigma_cm,
                 fmt='o', color='#c47c18', capsize=4, label="Observed", zorder=5)
axes[0].plot(r_model, los_best, '-', color='#3a4fc7', lw=2,
             label=f"Best fit: d={d_best:.1f} km, ΔV={dV_best:.3f} km³")
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set(xlabel="Distance from summit (km)",
            ylabel="LOS displacement (cm)",
            title=f"Best-fit Mogi model (RMS = {rms_best:.2f})")
axes[0].legend()

# Residuals
los_at_data = mogi_los_profile(dom_r_km, d_best, dV_best)
res_cm = dom_los_cm - los_at_data
axes[1].bar(dom_r_km, res_cm, width=1.2, color='#555', alpha=0.7)
axes[1].axhline(0, color='k', lw=1)
axes[1].fill_between([-1, 31], [-2, -2], [2, 2], color='#c47c18', alpha=0.15,
                      label="±2 cm uncertainty")
axes[1].set(xlabel="Distance from summit (km)",
            ylabel="Residual (cm)",
            title="Residuals: observed − model",
            xlim=[-1, 31])
axes[1].legend()

plt.tight_layout()
plt.show()


> **Interpret the inversion results:**
> 1. How does your best-fit Mogi depth compare to the first-order estimate from $r_{1/2}$ in Part 3a?
> 2. Inspect the residuals. Are they randomly distributed around zero, or is there a systematic pattern? What would a systematic pattern suggest about the Mogi source as a model for Domuyo?
> 3. The published model (Lundgren et al., 2020) uses an oblate sill (CDM) with a potency rate of 0.037 km³/yr. Your Mogi ΔV rate is different. Why do you expect the two values to differ, and in which direction?


---

## Part IV: The Depth–Volume Trade-off

Single-track InSAR cannot independently constrain depth and volume change from amplitude alone. The peak LOS displacement scales as $\Delta V / d^2$, so many (depth, $\Delta V$) pairs produce similar peak values. The shape of the radial profile — specifically its width — provides an independent depth constraint.

In this part, you will map the misfit surface in depth–volume space to visualize and characterize the trade-off.

### Prediction

Sketch what you expect the misfit contours to look like in (depth, $\Delta V$) space. Will the minimum be a point or an elongated valley? In what direction will the valley be oriented?


In [ ]:
# Grid search over depth and volume change
depth_grid = np.linspace(1.0, 15.0, 80)
dV_grid    = np.linspace(0.005, 0.250, 80)

DD, DV = np.meshgrid(depth_grid, dV_grid)
RMS    = np.zeros_like(DD)

for i in range(DD.shape[0]):
    for j in range(DD.shape[1]):
        RMS[i, j] = rms_misfit(DD[i, j], DV[i, j],
                                dom_r_km, dom_los_cm, dom_sigma_cm)

print(f"Grid search complete. Minimum RMS = {RMS.min():.3f} "
      f"at d = {DD.flat[RMS.argmin()]:.1f} km, ΔV = {DV.flat[RMS.argmin()]:.3f} km³")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Misfit surface
im = axes[0].contourf(DD, DV, RMS, levels=np.linspace(0, 5, 26), cmap='viridis_r')
axes[0].contour(DD, DV, RMS, levels=[1.0, 1.5, 2.0, 3.0], colors='white',
                linewidths=0.8, linestyles='--')
axes[0].plot(d_best, dV_best, 'r*', ms=14, label=f"Best fit ({d_best:.1f} km, {dV_best:.3f} km³)")
plt.colorbar(im, ax=axes[0], label="Normalized RMS misfit")
axes[0].set(xlabel="Depth (km)", ylabel="ΔV (km³)",
            title="Mogi inversion misfit surface")
axes[0].legend()

# Trade-off curve: at the best-fit depth, what range of ΔV gives RMS < 1.5?
# Also show the trade-off in the dV/d² space
ratio = DV / (DD**2)   # ΔV/d² — the quantity controlling peak LOS
peak_predicted = (1 - 0.25) / np.pi * DV * 1e9 / (DD * 1e3)**2 * np.cos(np.radians(36)) * 100

axes[1].scatter(DD[RMS < 2.0], DV[RMS < 2.0] * 1000, c=RMS[RMS < 2.0],
                cmap='viridis_r', s=10, vmin=0, vmax=2)
axes[1].plot(d_best, dV_best * 1000, 'r*', ms=14)

# Overlay constant-amplitude contour: ΔV/d² = const (the trade-off locus)
d_range = np.linspace(1, 15, 200)
best_ratio = dV_best / d_best**2
axes[1].plot(d_range, best_ratio * d_range**2 * 1000, 'k--', lw=1.5,
             label=r"$\Delta V \propto d^2$ (constant peak LOS)")
axes[1].set(xlabel="Depth (km)", ylabel="ΔV (km³ × 10⁻³)",
            title="Acceptable solutions (RMS < 2): the depth–ΔV trade-off")
axes[1].legend()

plt.tight_layout()
plt.show()


> **Interpret the misfit surface:**
> 1. Describe the shape of the misfit minimum. Is it a point or an elongated valley? In what direction is the valley oriented?
> 2. The dashed line in the right panel shows the locus where $\Delta V / d^2 = $ constant. How does this compare to the orientation of the misfit valley? What does this tell you about which combination of parameters is well-constrained by the data?
> 3. What additional measurement or observation would most effectively break the depth–volume trade-off? Justify your answer.
> 4. The acceptable region (RMS < 2) includes depths from roughly 4 to 10 km. What would you report as your best-fit depth and its uncertainty?


### 4b. The role of signal width in breaking the trade-off

Show that the signal width (half-amplitude radius) provides an independent constraint on depth.


In [ ]:
# For several depths along the trade-off valley, compute the best-fit ΔV
# and plot both the LOS profiles and the radial profiles

test_depths = [3.5, 5.0, 6.5, 8.0, 10.0]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for d in test_depths:
    # Best-fit ΔV for this fixed depth (1D minimization)
    result_1d = minimize(
        lambda x: rms_misfit(d, x[0], dom_r_km, dom_los_cm, dom_sigma_cm),
        x0=[dV_best], method='Nelder-Mead'
    )
    dV_this = result_1d.x[0]
    rms_this = result_1d.fun

    los_this = mogi_los_profile(r_model, d, dV_this)

    axes[0].plot(r_model, los_this, label=f"d={d} km, ΔV={dV_this:.3f} km³ (RMS={rms_this:.2f})")

axes[0].errorbar(dom_r_km, dom_los_cm, yerr=dom_sigma_cm,
                 fmt='o', color='#c47c18', capsize=4, label="Observed", zorder=5)
axes[0].axhline(0, color='k', lw=0.5)
axes[0].set(xlabel="Distance from summit (km)",
            ylabel="LOS displacement (cm)",
            title="Same data, different depths — LOS profiles")
axes[0].legend(fontsize=9)

# Half-amplitude radius for each model
d_arr   = np.linspace(1, 15, 100)
r12_arr = d_arr * np.sqrt(2**(2/3) - 1)
axes[1].plot(d_arr, r12_arr, 'k-', lw=2, label=r"$r_{1/2} \approx 0.77\,d$")

# Measured half-amplitude radius from data
axes[1].axhline(r_half_interp, color='#c47c18', ls='--', lw=2,
                label=f"Observed r½ ≈ {r_half_interp:.1f} km")
axes[1].axvline(d_estimate, color='#c47c18', ls=':', lw=1)

axes[1].set(xlabel="Source depth (km)",
            ylabel="Half-amplitude radius (km)",
            title="r½ as a depth estimator")
axes[1].legend()

plt.tight_layout()
plt.show()


> **Interpret the trade-off plots:**
> 1. All curves in the left panel pass near the data at small $r$ (they all match the peak LOS because $\Delta V$ was optimized). But how do they differ at large $r$ (beyond ~10 km)? Which depth is most consistent with the data there?
> 2. The right panel shows that the half-amplitude radius uniquely maps to a source depth. If you measured $r_{1/2} = 5$ km from an interferogram, what is your depth estimate? What if $r_{1/2} = 10$ km?
> 3. In practice, measurement noise limits how precisely you can read $r_{1/2}$ from an interferogram. How does this uncertainty propagate into depth uncertainty?


---

## Synthesis

> **Answer each question in one or two sentences:**
>
> 1. The Mogi source predicts that $r_{1/2} \approx 0.77\,d$. How would you use this relationship, along with the peak LOS displacement, to make a quick estimate of both source depth and volume change from an interferogram — without running any inversion?
>
> 2. The published Domuyo source model uses an oblate sill (CDM) rather than a Mogi sphere. How would you expect the LOS profile of a sill at the same depth and with the same peak displacement to differ from the Mogi prediction, and why?
>
> 3. You observe a bulls-eye interferogram at an unmonitored volcano. The signal is 15 cm peak LOS over 2 years, with a half-amplitude radius of ~8 km. Estimate (a) the source depth and (b) the volume change rate, stating all assumptions.
>
> 4. How does combining ascending and descending InSAR tracks help constrain the Mogi source, beyond what a single track provides?
>
> 5. The misfit valley in Part IV runs approximately along the direction of constant $\Delta V / d^2$. This means the data constrain a specific combination of depth and volume change. What geophysical quantity does $\Delta V / d^2$ represent, and what additional observation would you need to break this trade-off?


---

## Summary

- The Mogi (1958) point source predicts axially symmetric surface displacement: $u_z \propto d\,(r^2+d^2)^{-3/2}$ and $u_r \propto r\,(r^2+d^2)^{-3/2}$.
- Source depth controls signal width via the half-amplitude radius $r_{1/2} \approx 0.77\,d$. This relationship is independent of volume change and can be used for rapid depth estimation.
- InSAR measures LOS displacement: primarily vertical, with a horizontal contribution that creates left–right asymmetry in ascending and descending geometries.
- For the Domuyo interferogram (2015–2018), the best-fit Mogi source has depth ~6.5 km and ΔV rate ~0.026 km³/yr. The published CDM sill model gives a potency rate of 0.037 km³/yr at the same depth.
- The depth–volume trade-off means that peak LOS alone cannot separately constrain $d$ and $\Delta V$; fitting the full radial profile shape is essential.

### Data and software

Domuyo interferogram data adapted from Lundgren, P., et al. (2020). The dynamics of large silicic systems from satellite remote sensing observations: the intriguing case of Domuyo volcano, Argentina. *Scientific Reports*, 10, 11642. [https://doi.org/10.1038/s41598-020-67982-8](https://doi.org/10.1038/s41598-020-67982-8) (CC BY 4.0).

Mogi, K. (1958). Relations between the eruptions of various volcanoes and the deformations of the ground surfaces around them. *Bulletin of the Earthquake Research Institute*, University of Tokyo, 36, 99–134.

Segall, P. (2010). *Earthquake and Volcano Deformation*. Princeton University Press.
